In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-01_Template Processed Data.xlsx
/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-03_Template Processed Data.xlsx
/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/IC-02_Template Processed Data.xlsx
/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-02_Template Processed Data.xlsx
/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/IC-06_Template Processed Data.xlsx
/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/IC-01_Template Processed Data.xlsx
/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-04_Template Processed Data.xlsx
/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-06_Through_Template Processed Data.xlsx
/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-06_UTurn_Template Processed Data.xlsx
/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/Rev 1_IC-03_Template Processed Data.xlsx


In [2]:
import pandas as pd
import re

file_path_IC1 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/IC-01_Template Processed Data.xlsx"
file_path_IC2 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/IC-02_Template Processed Data.xlsx"
file_path_IC3 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/Rev 1_IC-03_Template Processed Data.xlsx"
file_path_IC4 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/IC-04_Template Processed Data.xlsx"
file_path_MC1 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-01_Template Processed Data.xlsx"
file_path_MC2 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-02_Template Processed Data.xlsx"
file_path_MC3 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-03_Template Processed Data.xlsx"
file_path_MC4 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-04_Template Processed Data.xlsx"
file_path_MC5 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-05_Template Processed Data.xlsx"
file_path_MC6 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-06_UTurn_Template Processed Data.xlsx"
file_path_MC7 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-07_Template Processed Data.xlsx"
file_path_MC8 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-08_Template Processed Data.xlsx"
file_path_MC9 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-09_Template Processed Data.xlsx"
file_path_MC10 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-10_Template Processed Data.xlsx"


In [6]:
df = pd.read_excel(file_path_IC4, sheet_name="Peak Hour_Combined")

# Rename first column to TimeLabel
df = df.rename(columns={df.columns[0]: "TimeLabel"})

# Keep only Thu rows (change to (Thu|Fri) if needed)
df = df[df["TimeLabel"].astype(str).str.match(r"^(Thu)\s*:\s*\d+", na=False)].copy()

# Extract hour from "Thu: 10"
df["Hour"] = df["TimeLabel"].astype(str).str.extract(r":\s*(\d+)").astype(int)

# Time blocks
def block(h):
    if 8 <= h <= 11:
        return "Morning"
    if 12 <= h <= 16:
        return "Noon"
    return "Evening"   # includes 17–24 and 1–7 if those exist

df["Block"] = df["Hour"].apply(block)

# Direction column pairs (explicit, correct)
pairs = []
for d in range(1, 13):
    vol_col = f"Direction-{d}"
    k_col = f"Direction-{d}.1"
    pairs.append((d, vol_col, k_col))

rows = []
for d, vol_col, k_col in pairs:
    for b in ["Morning", "Noon", "Evening"]:
        sub = df[df["Block"] == b].copy()
        sub[vol_col] = pd.to_numeric(sub[vol_col], errors="coerce")

        if sub[vol_col].notna().sum() == 0:
            rows.append([d, b, None, None, None])
            continue

        idx = sub[vol_col].idxmax()
        rows.append([
            d,
            b,
            df.loc[idx, "TimeLabel"],     # Time of max
            df.loc[idx, k_col],           # K-factor at max
            df.loc[idx, vol_col]          # Max volume
        ])

result = pd.DataFrame(rows, columns=["Direction", "Block", "Time_of_Max", "KFactor_of_Max", "Max_Volume"])
print(result)


    Direction    Block Time_of_Max  KFactor_of_Max  Max_Volume
0           1  Morning     Thu: 10        0.056077       293.0
1           1     Noon     Thu: 12        0.061818       323.0
2           1  Evening     Thu: 18        0.068517       358.0
3           2  Morning     Thu: 11        0.046397       179.0
4           2     Noon     Thu: 14        0.065578       253.0
5           2  Evening     Thu: 20        0.069207       267.0
6           3  Morning     Thu: 11        0.054550       470.0
7           3     Noon     Thu: 12        0.056987       491.0
8           3  Evening     Thu: 19        0.061165       527.0
9           4  Morning     Thu: 10        0.053932       692.0
10          4     Noon     Thu: 16        0.054555       700.0
11          4  Evening     Thu: 17        0.055257       709.0
12          5  Morning     Thu: 10        0.082040      2220.0
13          5     Noon     Thu: 16        0.069586      1883.0
14          5  Evening     Thu: 17        0.060273     

In [7]:
import pandas as pd

file_path = r"90daa560-bce4-4145-af07-e99ed515b01e.xlsx"

df = pd.read_excel(file_path_IC4, sheet_name="Peak Hour_Combined")
df = df.rename(columns={df.columns[0]: "TimeLabel"})

# Thursday only
df = df[df["TimeLabel"].astype(str).str.match(r"^(Thu)\s*:\s*\d+", na=False)].copy()

df["Hour"] = df["TimeLabel"].astype(str).str.extract(r":\s*(\d+)").astype(int)

def block(h):
    if 8 <= h <= 11:
        return "Morning"
    if 12 <= h <= 16:
        return "Noon"
    return "Evening"

df["Block"] = df["Hour"].apply(block)

pairs = [(d, f"Direction-{d}", f"Direction-{d}.1") for d in range(1, 13)]

rows = []
for d, vol_col, k_col in pairs:
    for b in ["Morning", "Noon", "Evening"]:
        sub = df[df["Block"] == b].copy()
        sub[vol_col] = pd.to_numeric(sub[vol_col], errors="coerce")

        if sub[vol_col].notna().sum() == 0:
            rows.append([d, b, None, None, None])
            continue

        idx = sub[vol_col].idxmax()

        # ✅ Convert K factor to percentage string like 5.61%
        k_pct = df.loc[idx, k_col] * 100
        k_pct_str = f"{k_pct:.2f}%"

        rows.append([
            d,
            b,
            df.loc[idx, "TimeLabel"],   # Time of max
            k_pct_str,                  # KFactor (%)
            df.loc[idx, vol_col]        # Max volume
        ])

result = pd.DataFrame(
    rows,
    columns=["Direction", "Block", "Time_of_Max", "KFactor (%)", "Max_Volume"]
)

result


,Direction,Block,Time_of_Max,KFactor (%),Max_Volume
0,1,Morning,Thu: 10,5.61%,293.0
1,1,Noon,Thu: 12,6.18%,323.0
2,1,Evening,Thu: 18,6.85%,358.0
3,2,Morning,Thu: 11,4.64%,179.0
4,2,Noon,Thu: 14,6.56%,253.0
5,2,Evening,Thu: 20,6.92%,267.0
6,3,Morning,Thu: 11,5.45%,470.0
7,3,Noon,Thu: 12,5.70%,491.0
8,3,Evening,Thu: 19,6.12%,527.0
9,4,Morning,Thu: 10,5.39%,692.0


In [8]:
# ✅ Save to Excel
output_file = r"IC4.xlsx"
result.to_excel(output_file, index=False)

print("Saved:", output_file)

Saved: IC4.xlsx


## Preferred Format

In [9]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

In [10]:
# -------------------- INPUT --------------------

import re

file_path_IC1 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/IC-01_Template Processed Data.xlsx"
file_path_IC2 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/IC-02_Template Processed Data.xlsx"
file_path_IC3 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/Rev 1_IC-03_Template Processed Data.xlsx"
file_path_IC4 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/IC-04_Template Processed Data.xlsx"
file_path_MC1 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-01_Template Processed Data.xlsx"
file_path_MC2 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-02_Template Processed Data.xlsx"
file_path_MC3 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-03_Template Processed Data.xlsx"
file_path_MC4 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-04_Template Processed Data.xlsx"
file_path_MC5 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-05_Template Processed Data.xlsx"
file_path_MC6 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-06_UTurn_Template Processed Data.xlsx"
file_path_MC7 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-07_Template Processed Data.xlsx"
file_path_MC8 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-08_Template Processed Data.xlsx"
file_path_MC9 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-09_Template Processed Data.xlsx"
file_path_MC10 = r"/kaggle/input/classified-vehicle-count-ctc-dataset-19-dec-2025/MC-10_Template Processed Data.xlsx"

sheet_name = "Peak Hour_Combined"

In [17]:
# -------------------- STEP 1: READ + FILTER (THURSDAY ONLY) --------------------
df = pd.read_excel(file_path_IC4, sheet_name=sheet_name)
df = df.rename(columns={df.columns[0]: "TimeLabel"})
df = df[df["TimeLabel"].astype(str).str.match(r"^(Thu)\s*:\s*\d+", na=False)].copy()

df["Hour"] = df["TimeLabel"].astype(str).str.extract(r":\s*(\d+)").astype(int)

def block(h):
    if 8 <= h <= 11:
        return "Morning"
    if 12 <= h <= 16:
        return "Noon"
    return "Evening"

df["Block"] = df["Hour"].apply(block)

# -------------------- STEP 2: FIND PEAK (VOL, TIME, K) FOR EACH DIRECTION & BLOCK --------------------
pairs = [(d, f"Direction-{d}", f"Direction-{d}.1") for d in range(1, 8)]

rows = []
for d, vol_col, k_col in pairs:
    for b in ["Morning", "Noon", "Evening"]:
        sub = df[df["Block"] == b].copy()
        sub[vol_col] = pd.to_numeric(sub[vol_col], errors="coerce")

        idx = sub[vol_col].idxmax()
        best_time = df.loc[idx, "TimeLabel"]
        best_vol = int(df.loc[idx, vol_col])
        best_k = df.loc[idx, k_col] * 100  # convert to percent

        rows.append([d, b, best_time, f"{best_k:.2f}%", best_vol])

long_df = pd.DataFrame(rows, columns=["Direction", "Block", "Time", "K Factor", "Volume"])

# -------------------- STEP 3: CONVERT TO WIDE FORMAT (LIKE YOUR SCREENSHOT) --------------------
blocks = ["Morning", "Noon", "Evening"]
wide_rows = []

for d in range(1, 13):
    r = {"Direction": f"Direction {d}"}
    for b in blocks:
        x = long_df[(long_df["Direction"] == d) & (long_df["Block"] == b)].iloc[0]
        r[f"{b}_Time"] = x["Time"]
        r[f"{b}_K"] = x["K Factor"]
        r[f"{b}_Vol"] = x["Volume"]
    wide_rows.append(r)

wide_df = pd.DataFrame(wide_rows)

wide_df

IndexError: single positional indexer is out-of-bounds

In [12]:
# -------------------- STEP 4: WRITE FORMATTED EXCEL --------------------
wb = Workbook()
ws = wb.active
ws.title = "IC4_Thursday_Peak_Summary"

bold = Font(bold=True)
center = Alignment(horizontal="center", vertical="center", wrap_text=True)
left = Alignment(horizontal="left", vertical="center", wrap_text=True)

thin = Side(style="thin", color="000000")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

fill_morning = PatternFill("solid", fgColor="FFC000")   # orange
fill_noon = PatternFill("solid", fgColor="00B0F0")      # blue
fill_evening = PatternFill("solid", fgColor="FFD966")   # yellow
fill_sub = PatternFill("solid", fgColor="D9D9D9")       # gray

# Block headers (merged)
ws.merge_cells("B1:D1")
ws["B1"] = "Morning (8 to 11)"
ws["B1"].font = bold
ws["B1"].alignment = center
ws["B1"].fill = fill_morning

ws.merge_cells("E1:G1")
ws["E1"] = "Noon (12-16)"
ws["E1"].font = bold
ws["E1"].alignment = center
ws["E1"].fill = fill_noon

ws.merge_cells("H1:J1")
ws["H1"] = "Evening (17 to 7)"
ws["H1"].font = bold
ws["H1"].alignment = center
ws["H1"].fill = fill_evening

# Subheaders row
subheaders = ["Time", "K Factor", "Volume"]
for start_col in [2, 5, 8]:
    for j, h in enumerate(subheaders):
        cell = ws.cell(row=2, column=start_col + j, value=h)
        cell.font = bold
        cell.alignment = center
        cell.fill = fill_sub

# Fill data
start_row = 3
for i, row in wide_df.iterrows():
    r = start_row + i

    c = ws.cell(row=r, column=1, value=row["Direction"])
    c.font = bold
    c.alignment = left

    ws.cell(r, 2, row["Morning_Time"]).alignment = center
    ws.cell(r, 3, row["Morning_K"]).alignment = center
    ws.cell(r, 4, row["Morning_Vol"]).alignment = center

    ws.cell(r, 5, row["Noon_Time"]).alignment = center
    ws.cell(r, 6, row["Noon_K"]).alignment = center
    ws.cell(r, 7, row["Noon_Vol"]).alignment = center

    ws.cell(r, 8, row["Evening_Time"]).alignment = center
    ws.cell(r, 9, row["Evening_K"]).alignment = center
    ws.cell(r, 10, row["Evening_Vol"]).alignment = center

# Borders + column widths
max_row = start_row + len(wide_df) - 1
max_col = 10

for rr in range(1, max_row + 1):
    for cc in range(1, max_col + 1):
        ws.cell(rr, cc).border = border

widths = {1: 16, 2: 10, 3: 10, 4: 10, 5: 10, 6: 10, 7: 10, 8: 10, 9: 10, 10: 10}
for c, w in widths.items():
    ws.column_dimensions[get_column_letter(c)].width = w

# -------------------- SAVE --------------------
output_file = "IC4_Preferred.xlsx"
wb.save(output_file)

print("Saved:", output_file)


Saved: IC4_Preferred.xlsx
